# The drum machine as a class
## Lab 05 · ACTAM 2026

In the last lesson a drum machine was built in the browser, in the style of the Roland TR-808: sixteen keys, a LED on each, a step that moves along. The code was split in three parts:

- the **model**: the data that describes the machine, `keys` (sixteen `true`/`false`) and `step`;
- the **controller**: the functions that change the model, such as a click that flips `keys[i]` or `nextStep`;
- the **view**: the function that draws the model, `render`.

Today we build the same machine in Python, with a **class**. A class keeps the model (its attributes) and the controller (its methods) together in one object. The views are ways of looking at that object: as text, as coloured keys, as sound.

### Setup

Run this cell first. It has `play`, three drum sounds in numpy, `mix` and `bar_html`, which draws a row of keys. Nothing in it is an exercise: read it if you are curious.

In [9]:
SR = 48000

import json
import numpy as np
from IPython.display import Audio, HTML, display

def play(wave):
    return Audio(np.clip(wave, -1, 1), rate=SR, normalize=False, autoplay=False)

rng = np.random.default_rng(808)

def decay(duration, tau):
    t = np.arange(round(duration * SR)) / SR
    return np.exp(-t / tau)

def fade_out(wave, fade_s=0.005):
    n = round(fade_s * SR)
    out = wave.copy()
    out[-n:] *= np.linspace(1, 0, n)
    return out

def kick(duration=0.3, amp=0.45):
    t = np.arange(round(duration * SR)) / SR
    f = 45 + 135 * np.exp(-t / 0.03)            # the pitch falls from 180 Hz to 45 Hz
    phase = 2 * np.pi * np.cumsum(f) / SR
    return fade_out(amp * np.sin(phase) * decay(duration, 0.07))

def snare(duration=0.2, amp=0.25):
    t = np.arange(round(duration * SR)) / SR
    body = np.sin(2 * np.pi * 190 * t) * decay(duration, 0.03)
    wires = rng.uniform(-1, 1, len(t)) * decay(duration, 0.05)
    return fade_out(amp * (0.5 * body + 0.7 * wires))

def hat(duration=0.06, amp=0.15):
    noise = rng.uniform(-1, 1, round(duration * SR))
    bright = np.diff(noise, prepend=0) / 2      # differences keep the high frequencies
    return fade_out(amp * bright * decay(duration, 0.012))

kit = {"kick": kick(), "snare": snare(), "hat": hat()}

def mix(waves):
    out = np.zeros(max([len(w) for w in waves]))
    for w in waves:
        out[:len(w)] += w
    return out

COLOURS = ["#d9402b", "#ec8a2d", "#f0c63f", "#f3efe4"]     # red, orange, yellow, white

def bar_html (steps, step=None):
    width = 26 if len(steps) <= 16 else 14
    keys = ""
    for key in range(len(steps)):
        color = COLOURS[key * 4 // len(steps)] 
        border = "#36c95a" if key == step else "#1e1e1e"
        led_color = "#ff2a14" if steps[key] else "#3a0d08"
        led = (f'<div style="background:{led_color}; position: absolute; width: 5px; height: 5px;'
        f'left:50%; transform:translateX(-50%); top:5px; border-radius:50%"> </div>')
        keys += (f'<div style = "width: {width}px; height:40px; background: {color}; position:relative; '
                 f'display: inline-block; margin: 2px; border-radius: 4px; border:3px solid {border}"> {led} </div>' 
                 )
    outer_div = (f'<div style = "display:inline-block; background: grey; padding: 2px"> {keys} </div>')

    return outer_div

display(HTML(bar_html([True, False, False, False] * 4)))

## A · The model

### A1 · A pattern from a string
**Core · Write a class**

In lesson 04 a pattern was a string: `"x"` for a hit, `"."` for a rest. Now it becomes an object.

Write a class `Pattern` whose constructor takes such a string and keeps the model in one attribute, `steps`: a list of booleans, `True` for a hit. Any other symbol raises a `ValueError` with a message of your own.

```python
kick_p = Pattern("x...x...x...x...")
kick_p.steps        # [True, False, False, False, True, ...]
```

In [30]:
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Neither x nor .")

In [31]:
kick_p = Pattern("x...x...x...x...")
print(kick_p.steps)

try:
    kick_p = Pattern("ox..x....")
except ValueError as e:
    print("Value Error: ", e)

[True, False, False, False, True, False, False, False, True, False, False, False, True, False, False, False]
Value Error:  Neither x nor .


### A2 · Text, length and steps
**Core · Add methods**

Evaluate `kick_p` alone in a cell: Python shows something like `<__main__.Pattern at 0x10a2f...>`. That tells us nothing.

Add four methods:

- `text()` gives back the string, `"x...x..."`;
- `__repr__` uses it to show `Pattern('x...x...')`;
- `__len__` makes `len(kick_p)` work: the number of steps;
- `__getitem__` makes `kick_p[i]` work: `True` if step `i` is a hit.

Each section writes the class again with all the methods so far, and the new ones are left for you. In a file you would keep one class and add to it; in a notebook, writing it again lets every section run on its own. After writing a class again, make its objects again: the old ones still belong to the old class.

In [10]:
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Neither x nor .")
    def text(self):
        out = ""
        for i in self.steps:
            if i:
                out += "x"
            else:
                out += "."
        return out
    def __repr__(self):
        return f"Pattern({self.text()})"
    def __getitem__(self, i):
        return self.steps[i]
    def __len__(self):
        return len(self.steps)


In [11]:
kick_p = Pattern("x...x...x...x...")
print(len(kick_p), kick_p[0], kick_p[1])
kick_p

16 True False


Pattern(x...x...x...x...)

## B · Two more views

### B1 · Keys on the screen
**Core · Add a method**

When the last value of a cell is an object with a method called `_repr_html_`, Jupyter calls it and shows the HTML it gives back, instead of the text of `__repr__`. It is the same idea as `__repr__`: a name that someone else calls for us.

The setup has `bar_html(steps)`, which makes the HTML of a row of keys from a list of booleans. Add `_repr_html_` to `Pattern`: one line.

In [12]:
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Neither x nor .")
    def text(self):
        out = ""
        for i in self.steps:
            if i:
                out += "x"
            else:
                out += "."
        return out
    def __repr__(self):
        return f"Pattern({self.text()})"
    def __getitem__(self, i):
        return self.steps[i]
    def __len__(self):
        return len(self.steps)
    def _repr_html_(self):
        return bar_html(self.steps)

In [35]:
kick_p = Pattern("x...x...x...x...")
kick_p

Pattern(x...x...x...x...)

The object is the same; we look at it in two ways. Neither view is in the model: the colours of the keys live in `bar_html`, not in `steps`. Change `COLOURS` and every pattern looks different, with the same model.

### B2 · The sound view
**Core · Write a function**

A third view is sound. Write `render(pattern, sound, step_s=0.125)`, a function outside the class:

1. a step lasts `n = round(step_s * SR)` samples;
2. start from `np.zeros(len(pattern) * n + len(sound))`: the extra samples let the last sound ring;
3. for every step `i` that is a hit, add `sound` into the slice that starts at sample `i * n`.

At `step_s=0.125`, sixteen steps last two seconds: a bar of sixteenth notes at 120 BPM.

In [13]:
def render (pattern, sound, step_s=0.125):
    n = round(step_s * SR)
    out = np.zeros(len(pattern)*n + len(sound))
    for i in range(len(pattern)):
        if pattern[i]:
            out[i*n:i*n + len(sound)] = sound
    return out

pattern = Pattern("x...x...x...x...")

play(render(pattern, kit["kick"]))

`render` uses only `len(pattern)` and `pattern[i]`: it works on any object that has those two methods. It does not need to know how `Pattern` stores its steps.

### B3 · A beat
**Core · Try**

Three patterns, three sounds, one `mix`. The beat below is the one of *Billie Jean*, from lesson 04. Change it into a beat of your own.

In [14]:
kick_p = Pattern("x.......x.......")
snare_p = Pattern("....x.......x...")
hat_p = Pattern("x.x.x.x.x.x.x.x.")

"""def mix(waves):
    out = np.zeros(max([len(w) for w in waves]))
    for w in waves:
        out[:len(w)] += w
    return out"""

beat = mix([render(kick_p, kit["kick"]), render(snare_p, kit["snare"]), render(hat_p, kit["hat"])])
play(beat)


## C · The controller

### C1 · Change the model
**Core · Add methods**

The controller changes the model. Add three methods:

- `toggle(i)` flips step `i`: what a click on a key did in the browser;
- `clear()` turns every step off;
- `mirror()` reverses the pattern, so that the last step comes first.

For `mirror`, a slice with step `-1` reads a list backwards: `[1, 2, 3][::-1]` is `[3, 2, 1]`.

These methods change `self.steps` and give back nothing.

In [15]:
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Neither x nor .")
    def text(self):
        out = ""
        for i in self.steps:
            if i:
                out += "x"
            else:
                out += "."
        return out
    def __repr__(self):
        return f"Pattern({self.text()})"
    def __getitem__(self, i):
        return self.steps[i]
    def __len__(self):
        return len(self.steps)
    def _repr_html_(self):
        return bar_html(self.steps)
    
    def toggle(self, i):
        self.steps[i] = not self.steps[i]

    def clear(self):
        self.steps = [False] * len(self.steps)

    def mirror(self):
        self.steps = self.steps[::-1]

In [16]:
p1 = Pattern("x...x...x...x...")
display(HTML(p1._repr_html_()))
p1.mirror()
display(HTML(p1._repr_html_()))
p1.toggle(0)
display(HTML(p1._repr_html_()))
p1.clear()
display(HTML(p1._repr_html_()))

Since `mirror` gives back nothing, `q = p.mirror()` makes `q` equal to `None`, and `p` is the one that changes. The operators of lesson 07 worked the other way: `a >> b` made a new clip and left `a` and `b` as they were. Both are good designs; a method should make clear which one it follows.

## D · Equal, or the same?

### D1 · Save and load
In the browser, saving was `store = keys.slice()`. With `store = keys` alone, saving did not work. Here is the same thing in Python.

In [17]:
p = Pattern("x...x...x...x...")
store = p              # save?

p.toggle(2)
display(store)         # it changed too
print(store is p)

Pattern(x.x.x...x...x...)

True


`store = p` does not copy anything. It gives a second name to the same object, as in JavaScript. `is` asks whether two names refer to the same object.

Here is a first attempt at a copy. Before running the next cell, guess: does `new` change when `p` changes?

`new` is a different object, but its `steps` is the same list as the one of `p`: one list with two names, one level down.

### D2 · A real copy, and equality
**Core · Add methods**

Add two methods:

- `copy()` gives back a new pattern with the same steps and a list of its own. There is a short way: `text()` and the constructor.
- `__eq__(self, other)` makes `==` compare the steps. Without it, `==` behaves like `is`.

Load copies again: with `p = store`, the next `toggle` would change what was saved. With a copy, save, load and reset are each one line, because the model is all in `steps`.

## E · Operators

### E1 · Then, again, together
**Core · Add methods**

As with clips in lesson 07, but on patterns:

| expression | meaning | method |
|---|---|---|
| `a >> b` | `a`, then `b` | `__rshift__` |
| `a * 3` | `a` three times | `__mul__` |
| `a \| b` | a hit where either has one | `__or__` |

All three give back a new pattern and leave `a` and `b` as they were. `text()` makes the first two short. For `|`, raise a `ValueError` if the two patterns have different lengths.

In [18]:
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Neither x nor .")
    def text(self):
        out = ""
        for i in self.steps:
            if i:
                out += "x"
            else:
                out += "."
        return out
    def __repr__(self):
        return f"Pattern({self.text()})"
    def __getitem__(self, i):
        return self.steps[i]
    def __len__(self):
        return len(self.steps)
    def _repr_html_(self):
        return bar_html(self.steps)
    
    def toggle(self, i):
        self.steps[i] = not self.steps[i]

    def clear(self):
        self.steps = [False] * len(self.steps)

    def mirror(self):
        self.steps = self.steps[::-1]

    def __rshift__(self, other):
        return Pattern(self.text() + other.text())

    def __mul__(self, n):
        return Pattern(self.text() * n)

    def __or__(self, other):
        if (len(other.steps) != len(self.steps)):
            raise ValueError("The two patterns' lengths don't match")
        out = ""
        for i in range (len(self.steps)):
            if self.steps[i] or other.steps[i]:
                out += 'x'
            else:
                out += '.'
        return Pattern(out)


In [19]:
p = Pattern("x...x...x...x...")
display(HTML(p._repr_html_()))
p1 = Pattern("x..xx..xx..xx..x")
display(HTML((p >> p1)._repr_html_()))
display(HTML((p * 3)._repr_html_()))
p2 = Pattern("....x...x...x...")
display(HTML((p | p1)._repr_html_()))

`basic * 3 >> fill` is `(basic * 3) >> fill`: `*` comes before `>>`, as in lesson 07.

### E2 · Four bars
**Core · Try**

Write four bars of a beat with operators: three bars of a groove and one with a fill. Play them.

In [43]:
# groove_k = Pattern("x.......x.......")
# fill_k = Pattern("x.......x.x.x.xx")
# kicks = groove_k * 3 >> fill_k

# snares = Pattern("....x.......x...") * 3 >> Pattern("....x...xxxxxxxx")
# hats = Pattern("x.x.x.x.x.x.x.x.") * 4

# display(kicks, snares, hats)
# play(mix([render(kicks, kit["kick"]),
#           render(snares, kit["snare"]),
#           render(hats, kit["hat"])]))

## F · Save the state, not the machine

### F1 · A pattern as JSON
**Core · Add methods**

The model can be saved as text, for a file, a database or a message, and read back to rebuild the same machine. A common format is JSON, and the `json` module turns lists, dictionaries, numbers and booleans into text and back:

```python
json.dumps({"steps": [True, False]})      # '{"steps": [true, false]}'
json.loads('{"steps": [true, false]}')    # {'steps': [True, False]}
```

Add two methods:

- `to_json()` gives back the JSON text of `{"steps": ...}`;
- `from_json(text)` builds a new pattern from such a text. It is called on the class, `Pattern.from_json(text)`, not on a pattern: we do not have one yet.

A method called on the class is marked with `@classmethod`. Its first argument is the class itself, called `cls`, so `cls("")` makes a new, empty pattern. It is another way to build an object.

In [3]:
# Pattern, version 7: to and from JSON
class Pattern:
    def __init__(self, text):
        self.steps = []
        for c in text:
            if c == "x":
                self.steps.append(True)
            elif c == ".":
                self.steps.append(False)
            else:
                raise ValueError("Use only x for a hit and . for a rest.")

    def text(self):
        out = ""
        for hit in self.steps:
            if hit:
                out += "x"
            else:
                out += "."
        return out

    def __repr__(self):
        return "Pattern('" + self.text() + "')"

    def __len__(self):
        return len(self.steps)

    def __getitem__(self, i):
        return self.steps[i]

    def _repr_html_(self):
        return bar_html(self.steps)

    def toggle(self, i):
        self.steps[i] = not self.steps[i]

    def clear(self):
        self.steps = [False] * len(self)

    def mirror(self):
        self.steps = self.steps[::-1]

    def copy(self):
        return Pattern(self.text())

    def __eq__(self, other):
        return self.steps == other.steps

    def __rshift__(self, other):
        return Pattern(self.text() + other.text())

    def __mul__(self, n):
        return Pattern(self.text() * n)

    def __or__(self, other):
        if len(self) != len(other):
            raise ValueError("Both patterns must have the same number of steps.")
        out = ""
        for i in range(len(self)):
            if self[i] or other[i]:
                out += "x"
            else:
                out += "."
        return Pattern(out)

    def to_json(self):
        return json.dumps({"steps": self.steps})

    @classmethod
    def from_json(cls, text):
        data = json.loads(text)
        new = cls("")
        new.steps = data["steps"]
        return new

In [45]:
kick_p = Pattern("x.......x.x.....")
display(HTML(kick_p._repr_html_()))
print(kick_p.steps)
text = kick_p.to_json()
print(text)

again = Pattern.from_json(text)   # recall classmethods @classmethod

display(HTML(again._repr_html_()))
print(again.steps)

print("\n", again == kick_p, again is kick_p)

[True, False, False, False, False, False, False, False, True, False, True, False, False, False, False, False]
{"steps": [true, false, false, false, false, false, false, false, true, false, true, false, false, false, false, false]}


[True, False, False, False, False, False, False, False, True, False, True, False, False, False, False, False]

 True False


Only the steps are saved: no colours, no sounds, no tempo. Anyone with this text and the class can rebuild the same pattern.

**Try:** copy the printed text into `Pattern.from_json('...')`, change one `false` into `true`, and look at the keys.

## G · Extensions

### G1 · A drum machine of patterns
**Extension · Write a class**

A drum machine holds a pattern per instrument, a tempo and the current step: the model of the last lesson, `keys` and `step`, for three tracks. An object that holds other objects is called **composition**.

Write `DrumMachine(tracks, step_s=0.125)`, where `tracks` is a dictionary from a name in `kit` to a `Pattern`. It keeps `tracks`, `step_s` and `step = 0`, and has:

- `length`, a property: the number of steps of the longest track;
- `tick()`, the controller of the step: one step on, back to 0 after the last;
- `render()`, the sound view: every track rendered with its sound, mixed;
- `_repr_html_()`, a row of keys per track, with the current step marked: `bar_html(steps, step)` draws its border green.

In [20]:
class DrumMachine:
    def __init__(self, tracks, step_s=0.125):
        self.tracks = tracks
        self.step_s = step_s
        self.step = 0

    @property
    def length(self):
        return max([len(p) for p in self.tracks.values()])

    def tick(self):
        self.step = (self.step + 1) % self.length

    def render(self):
        return mix([render(self.tracks[name], kit[name], self.step_s) for name in self.tracks])

    def _repr_html_(self):
        rows = ""
        for name in self.tracks:
            rows += ('<div style="display:flex; align-items:center">'
                     + '<code style="width:56px">' + name + '</code>'
                     + bar_html(self.tracks[name].steps, self.step) + '</div>')
        return rows

dm = DrumMachine({"kick": Pattern("x.......x.x....."),
                  "snare": Pattern("....x.......x..."),
                  "hat": Pattern("x.x.x.x.x.x.x.x.")})
for i in range(5):
    dm.tick()
display(HTML(dm._repr_html_()))

dm.tracks["snare"].toggle(6)
dm.tick()
display(HTML(dm._repr_html_()))

play(dm.render())

In [4]:
!pip install firebase-admin

In [23]:
# Connect to firebase and save the patterns

import firebase_admin
from firebase_admin import credentials, firestore

class DrumMachine:
    def __init__(self, tracks, step_s=0.125):
        self.tracks = tracks
        self.step_s = step_s
        self.step = 0

    @property
    def length(self):
        return max([len(p) for p in self.tracks.values()])

    def tick(self):
        self.step = (self.step + 1) % self.length

    def render(self):
        return mix([render(self.tracks[name], kit[name], self.step_s) for name in self.tracks])

    def _repr_html_(self):
        rows = ""
        for name in self.tracks:
            rows += ('<div style="display:flex; align-items:center">'
                     + '<code style="width:56px">' + name + '</code>'
                     + bar_html(self.tracks[name].steps, self.step) + '</div>')
        return rows

    # new
    def save_patterns(self, db):
        db.collection("patterns").document("drum_machine_3").set({
        name: pattern.steps for name, pattern in self.tracks.items()
    })

    # new
    def load_pattern(self, db):
        data = db.collection("patterns").document("drum_machine").get().to_dict()
        for name, steps in data.items():
            text = "".join("x" if s else "." for s in steps)
            self.tracks[name] = Pattern(text)
        return(DrumMachine(self.tracks))

# settings
if not firebase_admin._apps:
    cred = credentials.Certificate("./drum-machine-patterns3-firebase-adminsdk-fbsvc-3a574367d9.json")
    firebase_admin.initialize_app(cred)
db = firestore.client()

# create a new DM and save the state
dm = DrumMachine({"kick": Pattern("x.......x.x....."),
                  "snare": Pattern("....x.......x..."),
                  "hat": Pattern("x.x.....x.x.x.x.")})
display(HTML(dm._repr_html_()))
dm.save_patterns(db)


In [71]:
# create a new DM with empty tracks and load patterns from the database
dm = DrumMachine({"kick": Pattern(""),
                  "snare": Pattern(""),
                  "hat": Pattern("")})
dm.load_pattern(db)
dm

Change a step with `dm.tracks["snare"].toggle(15)` and look at `dm` again: the machine shows the change, because it holds the pattern itself, not a copy.

### G2 · Euclidean rhythms
**Extension · Write a function**

Spread `k` hits as evenly as possible over `n` steps: step `i` is a hit when `(i * k) % n < k`. With 3 hits over 8 steps this gives `x..x..x.`, the *tresillo* of Cuban music, also heard in much pop music. Many traditional rhythms come out of this rule.

Write `euclid(k, n)`, which gives back a `Pattern`. Then try `euclid(5, 8)`, `euclid(5, 16)` and `euclid(7, 12)`.

In [47]:
def euclid(k, n):
    out = ""
    for i in range(n):
        if (i * k) % n < k:
            out += "x"
        else:
            out += "."
    return Pattern(out)

display(HTML(euclid(5, 8)._repr_html_()))
display(HTML(euclid(5, 16)._repr_html_()))
display(HTML(euclid(7, 12)._repr_html_()))

play(mix([render(euclid(3, 8) * 4, kit["kick"]),
          render(euclid(5, 16) * 2, kit["hat"]),
          render(Pattern("....x...") * 4, kit["snare"])]))

### G3 · Three against four
**Extension · Try**

Twelve steps hold three groups of four and four groups of three. Play `Pattern("x...") * 3` on one sound and `Pattern("x..") * 4` on another, together. Which one sounds like the beat?

In [48]:
threes = Pattern("x..") * 4
fours = Pattern("x...") * 3
display(HTML(threes._repr_html_()))
display(HTML(fours._repr_html_()))
display(HTML((threes | fours)._repr_html_()))
play(mix([render(fours * 4, kit["kick"], step_s=0.1),
          render(threes * 4, kit["hat"], step_s=0.1)]))

### G4 · Your own keys
**Extension · Write a function**

Write `my_bar_html(steps)`, your own version of `bar_html`: a `<div>` per step with inline CSS, as in the last lesson. Other colours, other shapes, a different LED. Then try it with `display(HTML(my_bar_html(kick_p.steps)))`.

In [49]:
def my_bar_html(steps, step=None):
    width = 30 if len(steps) <= 16 else 18
    keys = ""
    for i in range(len(steps)):
        bg = "#2e6e4f" if steps[i] else "#2b2b2b"
        border = "#f3efe4" if i == step else "#111"
        keys += (f'<div style="width:{width}px; height:{width}px; margin:3px; '
                 f'display:inline-block; background:{bg}; border:2px solid {border}"></div>')
    return f'<div style="display:inline-block; background:#444; padding:6px;">{keys}</div>'

kick_p = Pattern("x.......x.x.....")
display(HTML(my_bar_html(kick_p.steps)))
display(HTML(my_bar_html(euclid(5, 16).steps, step=3)))

### G5 · A dataclass
**Extension · Try**

`@dataclass`, from lesson 07, writes `__init__`, `__repr__` and `__eq__` for us from the list of fields. Here is a pattern as a dataclass, with no `__eq__` written by hand. Do two such patterns with the same steps compare equal? Which methods of our `Pattern` could a dataclass not write for us?

In [50]:
from dataclasses import dataclass

@dataclass
class DataPattern:
    steps: list

a = DataPattern([True, False, False, False])
b = DataPattern([True, False, False, False])
print(a, a == b, a is b)

DataPattern(steps=[True, False, False, False]) True False
